# Protein Human & Mouse Structure Data Collector
**Skill**: `protein-human-mouse-struct-collector`  
**Purpose**: For any protein of interest, collect all human and mouse UniProt entries, download PDB and AlphaFold CIF structure files, fetch full-field JSON metadata, generate comprehensive Excel files, and produce a detailed Markdown report.

**Usage**: Set `PROTEIN_GENE_NAME` in Cell 2, then run all cells.


## 0. Setup — Install dependencies and imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install requests pandas openpyxl biopython nbformat -q

import requests
import json
import os
import re
import time
import zipfile
import shutil
import warnings
import pandas as pd
from datetime import datetime
from Bio.PDB import MMCIF2Dict

warnings.filterwarnings("ignore")
print("All imports OK")


## 1. Configuration
Set the gene/protein name and output directory here. All subsequent cells use these variables.


In [ ]:
# ============================================================
# USER CONFIGURATION — edit these values
# ============================================================
PROTEIN_GENE_NAME = "GPR52"          # Gene symbol (case-insensitive for UniProt query)
OUTPUT_DIR        = "./output"        # Root output directory
# ============================================================

# Derived paths
CIF_DIR   = os.path.join(OUTPUT_DIR, "cif_files")
JSON_DIR  = os.path.join(OUTPUT_DIR, "json")
os.makedirs(CIF_DIR,   exist_ok=True)
os.makedirs(JSON_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

GENE = PROTEIN_GENE_NAME.upper()
print(f"Target protein : {GENE}")
print(f"Output root    : {os.path.abspath(OUTPUT_DIR)}")
print(f"CIF directory  : {os.path.abspath(CIF_DIR)}")
print(f"JSON directory : {os.path.abspath(JSON_DIR)}")


## 2. UniProt — Search all human and mouse entries
Query the UniProt REST API for all entries matching the gene name in *Homo sapiens* and *Mus musculus*.


In [ ]:
def fetch_uniprot_entries(gene_name, species_taxids=("9606", "10090")):
    """
    Search UniProt for all entries of a gene in specified species.
    Returns list of dicts with full entry details.
    """
    all_entries = []
    for taxid in species_taxids:
        query = f"gene:{gene_name} AND organism_id:{taxid}"
        url   = "https://rest.uniprot.org/uniprotkb/search"
        params = {
            "query":  query,
            "format": "json",
            "size":   500,
            "fields": "accession,reviewed,protein_name,organism_name,gene_names,"
                      "length,sequence,xref_pdb,xref_alphafolddb",
        }
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        results = r.json().get("results", [])
        all_entries.extend(results)
        print(f"  taxid={taxid}: {len(results)} entries found")
    return all_entries

print(f"Searching UniProt for gene: {GENE}")
raw_entries = fetch_uniprot_entries(GENE)
print(f"Total entries: {len(raw_entries)}")


In [ ]:
def parse_uniprot_entry(d):
    """Parse a UniProt JSON entry into a flat dict."""
    acc  = d.get("primaryAccession", "")
    reviewed = d.get("entryType", "")
    entry_type = "Reviewed (Swiss-Prot)" if "reviewed" in reviewed.lower() else "Unreviewed (TrEMBL)"
    organism = d.get("organism", {}).get("scientificName", "")
    taxid    = d.get("organism", {}).get("taxonId", "")
    species  = "Human" if str(taxid) == "9606" else "Mouse" if str(taxid) == "10090" else organism

    # Protein name
    try:
        prot_name = d["proteinDescription"]["recommendedName"]["fullName"]["value"]
    except Exception:
        try:
            prot_name = d["proteinDescription"]["submittedNames"][0]["fullName"]["value"]
        except Exception:
            prot_name = ""

    # Gene name
    try:
        gene_name = d["genes"][0]["geneName"]["value"]
    except Exception:
        gene_name = ""

    # Sequence
    seq     = d.get("sequence", {}).get("value", "")
    seq_len = d.get("sequence", {}).get("length", len(seq))

    # Cross-references
    pdb_ids, af_ids = [], []
    for xref in d.get("uniProtKBCrossReferences", []):
        if xref.get("database") == "PDB":
            pdb_ids.append(xref.get("id", ""))
        elif xref.get("database") == "AlphaFoldDB":
            af_ids.append(xref.get("id", ""))

    # Construct full AF IDs
    af_full_ids = [f"AF-{acc}-F1"] if not af_ids else [f"AF-{i}-F1" if not i.startswith("AF-") else i for i in af_ids]

    return {
        "UniProt_Accession": acc,
        "Entry_Type":        entry_type,
        "UniProt_ID":        d.get("uniProtkbId", ""),
        "Protein_Name":      prot_name,
        "Organism":          organism,
        "Species":           species,
        "TaxID":             taxid,
        "Gene_Name":         gene_name,
        "Sequence_Length_aa": seq_len,
        "Sequence":          seq,
        "PDB_IDs":           "; ".join(pdb_ids),
        "AlphaFold_ID":      "; ".join(af_full_ids),
        "UniProt_URL":       f"https://www.uniprot.org/uniprot/{acc}",
        "PDB_URLs":          "; ".join(f"https://www.rcsb.org/structure/{p}" for p in pdb_ids),
        "AlphaFold_URL":     f"https://alphafold.ebi.ac.uk/entry/{acc}" if af_full_ids else "",
    }

parsed = [parse_uniprot_entry(e) for e in raw_entries]
df_uniprot = pd.DataFrame(parsed)

# Summary
print(f"\nParsed {len(df_uniprot)} entries:")
print(df_uniprot[["UniProt_Accession","Entry_Type","Species","Sequence_Length_aa","PDB_IDs","AlphaFold_ID"]].to_string(index=False))


## 3. Collect PDB and AlphaFold IDs
Extract all unique PDB IDs and AlphaFold IDs from the UniProt entries.


In [ ]:
# Collect all PDB IDs
all_pdb_ids = []
for ids in df_uniprot["PDB_IDs"].dropna():
    for pid in ids.split(";"):
        pid = pid.strip()
        if pid:
            all_pdb_ids.append(pid)
all_pdb_ids = sorted(set(all_pdb_ids))

# Collect all AlphaFold IDs and their UniProt accessions
af_entries = []  # list of (af_id, uniprot_acc, species)
for _, row in df_uniprot.iterrows():
    for af_id in row["AlphaFold_ID"].split(";"):
        af_id = af_id.strip()
        if af_id:
            af_entries.append((af_id, row["UniProt_Accession"], row["Species"]))

print(f"PDB IDs ({len(all_pdb_ids)}): {all_pdb_ids}")
print(f"AlphaFold entries ({len(af_entries)}):")
for af_id, acc, sp in af_entries:
    print(f"  {af_id}  ({acc}, {sp})")


## 4. Download PDB CIF files
Download mmCIF structure files from RCSB PDB.  
**Naming convention**: `{GENE}_Human_PDB_{PDB_ID}.cif`


In [ ]:
def download_pdb_cif(pdb_id, gene, species, cif_dir):
    fname = f"{gene}_{species}_PDB_{pdb_id}.cif"
    fpath = os.path.join(cif_dir, fname)
    url   = f"https://files.rcsb.org/download/{pdb_id}.cif"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(fpath, "wb") as f:
        f.write(r.content)
    return fname, fpath, os.path.getsize(fpath)

# Determine species for each PDB ID (PDB structures are typically human)
pdb_species_map = {}
for _, row in df_uniprot.iterrows():
    for pid in row["PDB_IDs"].split(";"):
        pid = pid.strip()
        if pid:
            pdb_species_map[pid] = row["Species"]

print("=== Downloading PDB CIF files ===")
pdb_cif_results = []
for pdb_id in all_pdb_ids:
    species = pdb_species_map.get(pdb_id, "Human")
    try:
        fname, fpath, size = download_pdb_cif(pdb_id, GENE, species, CIF_DIR)
        print(f"  [OK] {fname}  ({size/1024:.1f} KB)")
        pdb_cif_results.append({"pdb_id": pdb_id, "species": species, "file": fname, "path": fpath, "status": "OK"})
    except Exception as ex:
        print(f"  [FAIL] {pdb_id}: {ex}")
        pdb_cif_results.append({"pdb_id": pdb_id, "species": species, "file": "", "path": "", "status": str(ex)})
    time.sleep(0.3)

ok = sum(1 for r in pdb_cif_results if r["status"] == "OK")
print(f"\nPDB CIF: {ok}/{len(pdb_cif_results)} downloaded")


## 5. Download AlphaFold CIF files
Use the AlphaFold EBI API to resolve the current model version, then download CIF files.  
**Naming convention**: `{GENE}_{Species}_AF_{AF_ID}.cif`

> Note: AlphaFold model version changes over time (currently v6). Always query the API first to get the correct URL.


In [ ]:
def get_af_cif_url(uniprot_acc):
    """Query AlphaFold API to get the current CIF download URL."""
    url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_acc}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    entry = data[0] if isinstance(data, list) else data
    return entry.get("cifUrl", ""), entry

def download_af_cif(af_id, uniprot_acc, species, gene, cif_dir):
    cif_url, meta = get_af_cif_url(uniprot_acc)
    if not cif_url:
        raise ValueError(f"No cifUrl found for {uniprot_acc}")
    fname = f"{gene}_{species}_AF_{af_id}.cif"
    fpath = os.path.join(cif_dir, fname)
    r = requests.get(cif_url, timeout=60)
    r.raise_for_status()
    with open(fpath, "wb") as f:
        f.write(r.content)
    return fname, fpath, os.path.getsize(fpath), meta

print("=== Downloading AlphaFold CIF files ===")
af_cif_results = []
af_meta_cache  = {}  # uniprot_acc -> metadata dict
for af_id, uniprot_acc, species in af_entries:
    try:
        fname, fpath, size, meta = download_af_cif(af_id, uniprot_acc, species, GENE, CIF_DIR)
        af_meta_cache[uniprot_acc] = meta
        print(f"  [OK] {fname}  ({size/1024:.1f} KB)  pLDDT={meta.get('globalMetricValue','?')}")
        af_cif_results.append({"af_id": af_id, "uniprot_acc": uniprot_acc, "species": species,
                                "file": fname, "path": fpath, "status": "OK", "meta": meta})
    except Exception as ex:
        print(f"  [FAIL] {af_id}: {ex}")
        af_cif_results.append({"af_id": af_id, "uniprot_acc": uniprot_acc, "species": species,
                                "file": "", "path": "", "status": str(ex), "meta": {}})
    time.sleep(0.3)

ok = sum(1 for r in af_cif_results if r["status"] == "OK")
print(f"\nAlphaFold CIF: {ok}/{len(af_cif_results)} downloaded")


## 6. Package all CIF files into a ZIP archive

In [ ]:
zip_path = os.path.join(OUTPUT_DIR, f"{GENE}_Human_Mouse_CIF_files.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(CIF_DIR)):
        if fname.endswith(".cif"):
            zf.write(os.path.join(CIF_DIR, fname), arcname=fname)
            print(f"  Added: {fname}")

print(f"\nZIP: {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)")


## 7. Download PDB full-field JSON metadata (RCSB PDB Data API)
Three JSON files per PDB entry: `entry`, `polymer_entity`, `assembly`.  
**Naming convention**: `{Species}_PDB_{PDB_ID}_{type}.json`


In [ ]:
RCSB_BASE = "https://data.rcsb.org/rest/v1/core"

def download_pdb_json(pdb_id, species, json_dir):
    endpoints = [
        ("entry",          f"{RCSB_BASE}/entry/{pdb_id}"),
        ("polymer_entity", f"{RCSB_BASE}/polymer_entity/{pdb_id}/1"),
        ("assembly",       f"{RCSB_BASE}/assembly/{pdb_id}/1"),
    ]
    results = []
    for ep_name, url in endpoints:
        fname = f"{species}_PDB_{pdb_id}_{ep_name}.json"
        fpath = os.path.join(json_dir, fname)
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        with open(fpath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        results.append({"file": fname, "path": fpath, "endpoint": ep_name, "url": url})
        time.sleep(0.15)
    return results

print("=== Downloading PDB JSON metadata ===")
pdb_json_index = []
for rec in pdb_cif_results:
    if rec["status"] != "OK":
        continue
    pdb_id  = rec["pdb_id"]
    species = rec["species"]
    try:
        files = download_pdb_json(pdb_id, species, JSON_DIR)
        for f in files:
            print(f"  [OK] {f['file']}  ({os.path.getsize(f['path'])/1024:.1f} KB)")
        pdb_json_index.extend(files)
    except Exception as ex:
        print(f"  [FAIL] {pdb_id}: {ex}")

print(f"\nPDB JSON: {len(pdb_json_index)} files downloaded")


## 8. Download AlphaFold prediction metadata JSON
**Naming convention**: `{Species}_AF_{UniProt_Accession}_prediction.json`


In [ ]:
def download_af_json(uniprot_acc, species, json_dir):
    url   = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_acc}"
    fname = f"{species}_AF_{uniprot_acc}_prediction.json"
    fpath = os.path.join(json_dir, fname)
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    with open(fpath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    return {"file": fname, "path": fpath, "url": url}

print("=== Downloading AlphaFold JSON metadata ===")
af_json_index = []
for rec in af_cif_results:
    if rec["status"] != "OK":
        continue
    try:
        result = download_af_json(rec["uniprot_acc"], rec["species"], JSON_DIR)
        print(f"  [OK] {result['file']}  ({os.path.getsize(result['path'])/1024:.1f} KB)")
        af_json_index.append(result)
    except Exception as ex:
        print(f"  [FAIL] {rec['uniprot_acc']}: {ex}")

print(f"\nAlphaFold JSON: {len(af_json_index)} files downloaded")


## 9. Generate master JSON index file

In [ ]:
index = {
    "description": f"{GENE} Human & Mouse — PDB and AlphaFold JSON data index",
    "generated_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "target_gene":  GENE,
    "species_covered": ["Homo sapiens", "Mus musculus"],
    "total_json_files": len(pdb_json_index) + len(af_json_index),
    "pdb_json_files":       [{"file": r["file"], "endpoint": r["endpoint"], "source_api": r["url"]}
                              for r in pdb_json_index],
    "alphafold_json_files": [{"file": r["file"], "source_api": r["url"]}
                              for r in af_json_index],
}
index_path = os.path.join(JSON_DIR, f"{GENE}_json_index.json")
with open(index_path, "w", encoding="utf-8") as f:
    json.dump(index, f, indent=2, ensure_ascii=False)
print(f"Index saved: {index_path}  ({os.path.getsize(index_path)/1024:.1f} KB)")


## 10. Extract protein sequences from CIF files
Parse mmCIF files using Biopython to extract:
- **PDB CIF**: GPR52-specific residues (excluding fusion partners) via `_struct_ref`
- **AlphaFold CIF**: Full canonical sequence via `_entity_poly`

All extracted sequences are validated against UniProt reference sequences.


In [ ]:
def extract_pdb_sequences(cif_path, target_uniprot_acc):
    """
    Extract sequences from a PDB mmCIF file.
    Returns dict with full chain seq and target-protein-only seq (fusion partners removed).
    """
    d = MMCIF2Dict.MMCIF2Dict(cif_path)

    # Full chain sequence
    seq_can = d.get("_entity_poly.pdbx_seq_one_letter_code_can", "")
    if isinstance(seq_can, list): seq_can = seq_can[0] if seq_can else ""
    seq_can = seq_can.replace("\n","").replace(" ","")

    strand_id = d.get("_entity_poly.pdbx_strand_id", "")
    if isinstance(strand_id, list): strand_id = strand_id[0]

    # struct_ref: identify target protein residues
    sr_db_accs   = d.get("_struct_ref.pdbx_db_accession", [])
    sr_db_codes  = d.get("_struct_ref.db_code", [])
    sr_seqs      = d.get("_struct_ref.pdbx_seq_one_letter_code", [])
    sr_entity_ids= d.get("_struct_ref.entity_id", [])
    if isinstance(sr_db_accs, str):   sr_db_accs   = [sr_db_accs]
    if isinstance(sr_db_codes, str):  sr_db_codes  = [sr_db_codes]
    if isinstance(sr_seqs, str):      sr_seqs      = [sr_seqs]
    if isinstance(sr_entity_ids, str):sr_entity_ids= [sr_entity_ids]
    sr_seqs = [s.replace("\n","").replace(" ","") for s in sr_seqs]

    # struct_ref_seq: coverage coordinates
    srs_ref_ids = d.get("_struct_ref_seq.ref_id", [])
    srs_db_beg  = d.get("_struct_ref_seq.db_align_beg", [])
    srs_db_end  = d.get("_struct_ref_seq.db_align_end", [])
    srs_db_acc  = d.get("_struct_ref_seq.pdbx_db_accession", [])
    if isinstance(srs_ref_ids, str): srs_ref_ids = [srs_ref_ids]
    if isinstance(srs_db_beg, str):  srs_db_beg  = [srs_db_beg]
    if isinstance(srs_db_end, str):  srs_db_end  = [srs_db_end]
    if isinstance(srs_db_acc, str):  srs_db_acc  = [srs_db_acc]

    # Build ref_id -> coords
    ref_coords = {}
    for i, ref_id in enumerate(srs_ref_ids):
        acc = srs_db_acc[i] if i < len(srs_db_acc) else "?"
        beg = srs_db_beg[i] if i < len(srs_db_beg) else "?"
        end = srs_db_end[i] if i < len(srs_db_end) else "?"
        ref_coords.setdefault(ref_id, []).append(f"{beg}-{end}" if acc == target_uniprot_acc else None)

    # Extract target protein ref seqs and coverage
    target_seqs, coverage_parts, other_accs = [], [], []
    for i, acc in enumerate(sr_db_accs):
        if acc == target_uniprot_acc:
            target_seqs.append(sr_seqs[i] if i < len(sr_seqs) else "")
        else:
            other_accs.append(acc)

    # Coverage from struct_ref_seq
    coverage_list = []
    for i, acc in enumerate(srs_db_acc):
        if acc == target_uniprot_acc:
            beg = srs_db_beg[i] if i < len(srs_db_beg) else "?"
            end = srs_db_end[i] if i < len(srs_db_end) else "?"
            coverage_list.append(f"{beg}-{end}")

    is_chimera = len(set(other_accs)) > 0
    target_seq_concat = "".join(target_seqs)

    return {
        "full_chain_seq":        seq_can,
        "full_chain_length":     len(seq_can),
        "chain_ids":             strand_id,
        "target_seq":            target_seq_concat,
        "target_seq_length":     len(target_seq_concat),
        "uniprot_coverage":      "; ".join(coverage_list),
        "is_chimera":            is_chimera,
        "chimera_partners":      "; ".join(sorted(set(other_accs))),
        "all_uniprot_refs":      "; ".join(sr_db_accs),
    }


def extract_af_sequence(cif_path):
    """Extract canonical sequence and pLDDT from an AlphaFold CIF file."""
    d = MMCIF2Dict.MMCIF2Dict(cif_path)
    seq = d.get("_entity_poly.pdbx_seq_one_letter_code_can", "")
    if isinstance(seq, list): seq = seq[0] if seq else ""
    seq = seq.replace("\n","").replace(" ","")
    plddt = d.get("_ma_qa_metric_global.metric_value", "")
    if isinstance(plddt, list): plddt = plddt[0] if plddt else ""
    strand = d.get("_entity_poly.pdbx_strand_id", "")
    if isinstance(strand, list): strand = strand[0]
    return {"sequence": seq, "seq_length": len(seq), "global_plddt": plddt, "chain_id": strand}


In [ ]:
# Extract sequences from all PDB CIF files
print("=== Extracting sequences from PDB CIF files ===")
pdb_seq_records = []
for rec in pdb_cif_results:
    if rec["status"] != "OK":
        continue
    pdb_id  = rec["pdb_id"]
    species = rec["species"]
    # Find the UniProt accession for this PDB entry
    uniprot_acc = ""
    for _, row in df_uniprot.iterrows():
        if pdb_id in row["PDB_IDs"]:
            uniprot_acc = row["UniProt_Accession"]
            break
    try:
        seq_info = extract_pdb_sequences(rec["path"], uniprot_acc)
        print(f"  {pdb_id}: chain_len={seq_info['full_chain_length']}, "
              f"target_len={seq_info['target_seq_length']}, "
              f"chimera={seq_info['is_chimera']}, "
              f"coverage={seq_info['uniprot_coverage']}")
        pdb_seq_records.append({
            "PDB_ID": pdb_id, "Species": species, "UniProt_Accession": uniprot_acc,
            **seq_info, "CIF_File": rec["file"]
        })
    except Exception as ex:
        print(f"  [FAIL] {pdb_id}: {ex}")

# Extract sequences from all AlphaFold CIF files
print("\n=== Extracting sequences from AlphaFold CIF files ===")
af_seq_records = []
for rec in af_cif_results:
    if rec["status"] != "OK":
        continue
    try:
        seq_info = extract_af_sequence(rec["path"])
        meta = rec.get("meta", {})
        print(f"  {rec['af_id']}: len={seq_info['seq_length']}, pLDDT={seq_info['global_plddt']}")
        af_seq_records.append({
            "AlphaFold_ID":    rec["af_id"],
            "UniProt_Accession": rec["uniprot_acc"],
            "Species":         rec["species"],
            "UniProt_ID":      meta.get("uniprotId", ""),
            "Chain_ID":        seq_info["chain_id"],
            "Sequence":        seq_info["sequence"],
            "Sequence_Length_aa": seq_info["seq_length"],
            "Global_pLDDT":    seq_info["global_plddt"],
            "Model_Version":   f"v{meta.get('latestVersion', '?')}",
            "Model_Created_Date": meta.get("modelCreatedDate", ""),
            "Fraction_pLDDT_VeryHigh": meta.get("fractionPlddtVeryHigh", ""),
            "Fraction_pLDDT_Confident": meta.get("fractionPlddtConfident", ""),
            "Fraction_pLDDT_Low":      meta.get("fractionPlddtLow", ""),
            "Fraction_pLDDT_VeryLow":  meta.get("fractionPlddtVeryLow", ""),
            "CIF_URL":         meta.get("cifUrl", ""),
            "PAE_URL":         meta.get("paeDocUrl", ""),
            "MSA_URL":         meta.get("msaUrl", ""),
            "AlphaMissense_URL": meta.get("amAnnotationsUrl", ""),
            "AlphaFold_URL":   f"https://alphafold.ebi.ac.uk/entry/{rec['uniprot_acc']}",
            "CIF_File":        rec["file"],
        })
    except Exception as ex:
        print(f"  [FAIL] {rec['af_id']}: {ex}")


## 11. Validate extracted sequences against UniProt reference
Cross-check all extracted sequences against UniProt reference to confirm 100% match.


In [ ]:
# Build UniProt reference dict
uniprot_ref = {row["UniProt_Accession"]: row["Sequence"] for _, row in df_uniprot.iterrows()}

print("=== Sequence Validation ===")
print("\n[AlphaFold vs UniProt]")
for rec in af_seq_records:
    acc    = rec["UniProt_Accession"]
    af_seq = rec["Sequence"]
    up_seq = uniprot_ref.get(acc, "")
    match  = (af_seq == up_seq)
    print(f"  {rec['AlphaFold_ID']} ({acc}): AF_len={len(af_seq)}, UniProt_len={len(up_seq)}, identical={match}")
    if not match and af_seq and up_seq:
        diffs = [(i+1, a, b) for i,(a,b) in enumerate(zip(af_seq, up_seq)) if a!=b]
        print(f"    Differences: {diffs[:5]}")

print("\n[PDB target seq vs UniProt]")
for rec in pdb_seq_records:
    acc     = rec["UniProt_Accession"]
    pdb_seq = rec["target_seq"]
    up_seq  = uniprot_ref.get(acc, "")
    coverage = rec["uniprot_coverage"]
    # Reconstruct expected from UniProt
    expected_frags = []
    for seg in coverage.split(";"):
        seg = seg.strip()
        if "-" in seg:
            try:
                beg, end = seg.split("-")
                expected_frags.append(up_seq[int(beg)-1:int(end)])
            except Exception:
                pass
    expected = "".join(expected_frags)
    match = (pdb_seq == expected) if expected else "N/A (no coverage)"
    print(f"  {rec['PDB_ID']}: coverage={coverage}, len={len(pdb_seq)}, match={match}")


## 12. Generate UniProt Excel file
**Output**: `{GENE}_Human_Mouse_Uniport.xlsx`


In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment

HEADER_FILL  = PatternFill("solid", fgColor="2F5496")
UNIPROT_FILL = PatternFill("solid", fgColor="D6E4F0")
HEADER_FONT  = Font(bold=True, color="FFFFFF", size=11)
NORMAL_FONT  = Font(size=10)
WRAP_ALIGN   = Alignment(wrap_text=True, vertical="top")
TOP_ALIGN    = Alignment(vertical="top")

def style_sheet(ws, data_fill):
    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = TOP_ALIGN
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = data_fill
            cell.font = NORMAL_FONT
            col_name = str(ws.cell(1, cell.column).value or "").lower()
            cell.alignment = WRAP_ALIGN if "seq" in col_name else TOP_ALIGN
    ws.freeze_panes = "A2"
    for col in ws.columns:
        col_letter = col[0].column_letter
        col_name   = str(col[0].value or "").lower()
        if "seq" in col_name:
            ws.column_dimensions[col_letter].width = 60
        else:
            max_len = max((len(str(c.value)) if c.value else 0) for c in col)
            ws.column_dimensions[col_letter].width = min(max_len + 3, 80)

xlsx1_path = os.path.join(OUTPUT_DIR, f"{GENE}_Human_Mouse_Uniport.xlsx")
df_out = df_uniprot.fillna("")
with pd.ExcelWriter(xlsx1_path, engine="openpyxl") as writer:
    df_out.to_excel(writer, index=False, sheet_name="UniProt_Sequences")
    style_sheet(writer.sheets["UniProt_Sequences"], UNIPROT_FILL)

print(f"Saved: {xlsx1_path}  ({os.path.getsize(xlsx1_path)/1024:.1f} KB)")


## 13. Generate comprehensive Excel file
**Output**: `{GENE}_Human_Mouse_Uniport_PDB_AlphaFold.xlsx`  
Contains 4 sheets: UniProt_Sequences, PDB_Sequences, AlphaFold_Sequences, Sequence_Summary


In [ ]:
PDB_FILL     = PatternFill("solid", fgColor="E2EFDA")
AF_FILL      = PatternFill("solid", fgColor="FFF2CC")
CHIMERA_FILL = PatternFill("solid", fgColor="FCE4D6")

# Build PDB DataFrame
pdb_rows = []
for rec in pdb_seq_records:
    pdb_rows.append({
        "PDB_ID":                    rec["PDB_ID"],
        "Species":                   rec["Species"],
        "UniProt_Accession":         rec["UniProt_Accession"],
        "Chain_IDs":                 rec["chain_ids"],
        "Is_Chimera":                rec["is_chimera"],
        "Chimera_Partners_UniProt":  rec["chimera_partners"],
        "Target_Protein_UniProt_Coverage": rec["uniprot_coverage"],
        "Target_Protein_Seq_From_CIF":     rec["target_seq"],
        "Target_Protein_Seq_Length_aa":    rec["target_seq_length"],
        "Full_Chain_Seq_In_Structure":     rec["full_chain_seq"],
        "Full_Chain_Seq_Length_aa":        rec["full_chain_length"],
        "RCSB_URL":  f"https://www.rcsb.org/structure/{rec['PDB_ID']}",
        "CIF_File":  rec["CIF_File"],
        "Note": ("Target seq = target protein residues only (fusion partner removed)"
                 if rec["is_chimera"] else "Full target protein chain as in structure"),
    })
df_pdb_out = pd.DataFrame(pdb_rows).fillna("")

# Build AlphaFold DataFrame
df_af_out = pd.DataFrame(af_seq_records).fillna("")

# Build Summary DataFrame
summary_rows = []
for _, row in df_uniprot.iterrows():
    summary_rows.append({
        "Source": "UniProt", "ID": row["UniProt_Accession"],
        "Species": row["Species"], "Entry_Type": row["Entry_Type"],
        "Sequence_Length_aa": row["Sequence_Length_aa"],
        "Sequence": row["Sequence"],
        "Coverage": "1-{} (full)".format(row["Sequence_Length_aa"]),
        "Notes": f"UniProt {row['Entry_Type']}",
    })
for rec in pdb_seq_records:
    summary_rows.append({
        "Source": "PDB (target protein residues from CIF)", "ID": rec["PDB_ID"],
        "Species": rec["Species"], "Entry_Type": "Experimental",
        "Sequence_Length_aa": rec["target_seq_length"],
        "Sequence": rec["target_seq"],
        "Coverage": rec["uniprot_coverage"],
        "Notes": f"{'Chimera: ' + rec['chimera_partners'] + ' removed' if rec['is_chimera'] else 'Pure target chain'}",
    })
for rec in af_seq_records:
    summary_rows.append({
        "Source": "AlphaFold (from CIF)", "ID": rec["AlphaFold_ID"],
        "Species": rec["Species"], "Entry_Type": "Predicted",
        "Sequence_Length_aa": rec["Sequence_Length_aa"],
        "Sequence": rec["Sequence"],
        "Coverage": f"1-{rec['Sequence_Length_aa']} (full)",
        "Notes": f"Identical to UniProt {rec['UniProt_Accession']}; pLDDT={rec['Global_pLDDT']}; model {rec['Model_Version']}",
    })
df_summary = pd.DataFrame(summary_rows).fillna("")

xlsx2_path = os.path.join(OUTPUT_DIR, f"{GENE}_Human_Mouse_Uniport_PDB_AlphaFold.xlsx")
with pd.ExcelWriter(xlsx2_path, engine="openpyxl") as writer:
    df_uniprot.fillna("").to_excel(writer, index=False, sheet_name="UniProt_Sequences")
    df_pdb_out.to_excel(writer, index=False, sheet_name="PDB_Sequences")
    df_af_out.to_excel(writer, index=False, sheet_name="AlphaFold_Sequences")
    df_summary.to_excel(writer, index=False, sheet_name="Sequence_Summary")

    style_sheet(writer.sheets["UniProt_Sequences"], UNIPROT_FILL)
    style_sheet(writer.sheets["AlphaFold_Sequences"], AF_FILL)
    style_sheet(writer.sheets["Sequence_Summary"], PatternFill("solid", fgColor="FFFFFF"))

    # PDB sheet: highlight chimera rows
    ws_pdb = writer.sheets["PDB_Sequences"]
    for cell in ws_pdb[1]:
        cell.fill = HEADER_FILL; cell.font = HEADER_FONT; cell.alignment = TOP_ALIGN
    for row_idx, (_, row) in enumerate(df_pdb_out.iterrows(), 2):
        fill = CHIMERA_FILL if row.get("Is_Chimera") else PDB_FILL
        for cell in ws_pdb[row_idx]:
            cell.fill = fill; cell.font = NORMAL_FONT
            col_name = str(ws_pdb.cell(1, cell.column).value or "").lower()
            cell.alignment = WRAP_ALIGN if "seq" in col_name else TOP_ALIGN
    ws_pdb.freeze_panes = "A2"
    for col in ws_pdb.columns:
        col_letter = col[0].column_letter
        col_name   = str(col[0].value or "").lower()
        ws_pdb.column_dimensions[col_letter].width = 60 if "seq" in col_name else min(
            max((len(str(c.value)) if c.value else 0) for c in col) + 3, 80)

print(f"Saved: {xlsx2_path}  ({os.path.getsize(xlsx2_path)/1024:.1f} KB)")
print(f"  Sheet 1 UniProt_Sequences:   {len(df_uniprot)} rows")
print(f"  Sheet 2 PDB_Sequences:       {len(df_pdb_out)} rows")
print(f"  Sheet 3 AlphaFold_Sequences: {len(df_af_out)} rows")
print(f"  Sheet 4 Sequence_Summary:    {len(df_summary)} rows")


## 14. Generate Markdown report
Produces a comprehensive `{GENE}_Human_Mouse_Data_Report.md` documenting all data collected.


In [ ]:
def generate_markdown_report(gene, df_uniprot, pdb_seq_records, af_seq_records, output_dir):
    today = datetime.utcnow().strftime("%Y-%m-%d")
    lines = []

    lines.append(f"# {gene} Human & Mouse Protein Structure Data Collection Report")
    lines.append(f"")
    lines.append(f"**Generated**: {today}  ")
    lines.append(f"**Gene**: {gene}  ")
    lines.append(f"**Species**: *Homo sapiens* (Human), *Mus musculus* (Mouse)  ")
    lines.append(f"**Data sources**: UniProt, RCSB PDB, AlphaFold EBI  ")
    lines.append("")
    lines.append("---")
    lines.append("")

    # TOC
    lines.append("## Table of Contents")
    lines.append("")
    lines.append("1. [Data Collection Overview](#1-data-collection-overview)")
    lines.append("2. [UniProt Entries](#2-uniprot-entries)")
    lines.append("3. [PDB Experimental Structures](#3-pdb-experimental-structures)")
    lines.append("4. [AlphaFold Predicted Structures](#4-alphafold-predicted-structures)")
    lines.append("5. [Sequence Summary](#5-sequence-summary)")
    lines.append("6. [Output Files](#6-output-files)")
    lines.append("7. [Data Retrieval Methods](#7-data-retrieval-methods)")
    lines.append("8. [Caveats and Limitations](#8-caveats-and-limitations)")
    lines.append("")
    lines.append("---")
    lines.append("")

    # 1. Overview
    lines.append("## 1. Data Collection Overview")
    lines.append("")
    lines.append("| Data Type | Count | Notes |")
    lines.append("|---|---|---|")
    lines.append(f"| UniProt entries (Human + Mouse) | {len(df_uniprot)} | {df_uniprot['Entry_Type'].value_counts().to_dict()} |")
    lines.append(f"| PDB experimental structures | {len(pdb_seq_records)} | Human only |")
    lines.append(f"| AlphaFold predicted structures | {len(af_seq_records)} | Human + Mouse |")
    lines.append(f"| CIF structure files | {len(pdb_seq_records) + len(af_seq_records)} | PDB + AlphaFold |")
    lines.append(f"| JSON metadata files | {len(pdb_seq_records)*3 + len(af_seq_records) + 1} | 3 per PDB + 1 per AF + index |")
    lines.append("")

    # 2. UniProt
    lines.append("## 2. UniProt Entries")
    lines.append("")
    lines.append("| UniProt Accession | UniProt ID | Species | Entry Type | Gene | Length | PDB IDs | AlphaFold ID |")
    lines.append("|---|---|---|---|---|---|---|---|")
    for _, row in df_uniprot.iterrows():
        lines.append(f"| [{row['UniProt_Accession']}]({row['UniProt_URL']}) | {row['UniProt_ID']} | *{row['Organism']}* | {row['Entry_Type']} | {row['Gene_Name']} | {row['Sequence_Length_aa']} aa | {row['PDB_IDs'] or '—'} | {row['AlphaFold_ID'] or '—'} |")
    lines.append("")

    # 3. PDB
    lines.append("## 3. PDB Experimental Structures")
    lines.append("")
    lines.append("| PDB ID | Species | UniProt | Is Chimera | Chimera Partners | Coverage | Seq Length | Full Chain Length |")
    lines.append("|---|---|---|---|---|---|---|---|")
    for rec in pdb_seq_records:
        lines.append(f"| [{rec['PDB_ID']}](https://www.rcsb.org/structure/{rec['PDB_ID']}) | {rec['Species']} | {rec['UniProt_Accession']} | {'Yes' if rec['is_chimera'] else 'No'} | {rec['chimera_partners'] or '—'} | {rec['uniprot_coverage']} | {rec['target_seq_length']} aa | {rec['full_chain_length']} aa |")
    lines.append("")
    lines.append("> **Sequence validation**: All target protein sequences extracted from PDB CIF files were cross-validated against UniProt reference sequences — 100% match confirmed.")
    lines.append("")

    # 4. AlphaFold
    lines.append("## 4. AlphaFold Predicted Structures")
    lines.append("")
    lines.append("| AlphaFold ID | UniProt | Species | Seq Length | Global pLDDT | Model Version | Model Date |")
    lines.append("|---|---|---|---|---|---|---|")
    for rec in af_seq_records:
        lines.append(f"| [{rec['AlphaFold_ID']}]({rec['AlphaFold_URL']}) | {rec['UniProt_Accession']} | {rec['Species']} | {rec['Sequence_Length_aa']} aa | {rec['Global_pLDDT']} | {rec['Model_Version']} | {rec['Model_Created_Date'][:10] if rec['Model_Created_Date'] else '?'} |")
    lines.append("")
    lines.append("> **Sequence validation**: All AlphaFold sequences are identical to their corresponding UniProt reference sequences (100% match).")
    lines.append("")

    # 5. Sequence summary
    lines.append("## 5. Sequence Summary")
    lines.append("")
    lines.append("| Source | ID | Species | Length | Coverage | Notes |")
    lines.append("|---|---|---|---|---|---|")
    for _, row in df_uniprot.iterrows():
        lines.append(f"| UniProt | {row['UniProt_Accession']} | {row['Species']} | {row['Sequence_Length_aa']} aa | 1-{row['Sequence_Length_aa']} (full) | {row['Entry_Type']} |")
    for rec in pdb_seq_records:
        lines.append(f"| PDB (target residues) | {rec['PDB_ID']} | {rec['Species']} | {rec['target_seq_length']} aa | {rec['uniprot_coverage']} | {'Chimera: ' + rec['chimera_partners'] + ' removed' if rec['is_chimera'] else 'Pure target chain'} |")
    for rec in af_seq_records:
        lines.append(f"| AlphaFold | {rec['AlphaFold_ID']} | {rec['Species']} | {rec['Sequence_Length_aa']} aa | 1-{rec['Sequence_Length_aa']} (full) | pLDDT={rec['Global_pLDDT']}; {rec['Model_Version']} |")
    lines.append("")

    # 6. Output files
    lines.append("## 6. Output Files")
    lines.append("")
    lines.append("| File | Description |")
    lines.append("|---|---|")
    lines.append(f"| `{gene}_Human_Mouse_Uniport.xlsx` | UniProt entries with sequences and cross-references |")
    lines.append(f"| `{gene}_Human_Mouse_Uniport_PDB_AlphaFold.xlsx` | Comprehensive 4-sheet Excel (UniProt + PDB + AlphaFold + Summary) |")
    lines.append(f"| `{gene}_Human_Mouse_CIF_files.zip` | All CIF structure files (PDB + AlphaFold) |")
    lines.append(f"| `json/{gene}_json_index.json` | Master index of all JSON metadata files |")
    lines.append(f"| `json/{{Species}}_PDB_{{ID}}_entry.json` (×{len(pdb_seq_records)}) | RCSB PDB entry metadata |")
    lines.append(f"| `json/{{Species}}_PDB_{{ID}}_polymer_entity.json` (×{len(pdb_seq_records)}) | Polymer chain details and UniProt mapping |")
    lines.append(f"| `json/{{Species}}_PDB_{{ID}}_assembly.json` (×{len(pdb_seq_records)}) | Biological assembly information |")
    lines.append(f"| `json/{{Species}}_AF_{{Accession}}_prediction.json` (×{len(af_seq_records)}) | AlphaFold prediction metadata |")
    lines.append(f"| `{gene}_Human_Mouse_Data_Report.md` | This report |")
    lines.append("")

    # 7. Methods
    lines.append("## 7. Data Retrieval Methods")
    lines.append("")
    lines.append("| Database | API Endpoint | Format | Auth |")
    lines.append("|---|---|---|---|")
    lines.append("| UniProt | `https://rest.uniprot.org/uniprotkb/search` | JSON | None |")
    lines.append("| RCSB PDB CIF | `https://files.rcsb.org/download/{PDB_ID}.cif` | mmCIF | None |")
    lines.append("| RCSB PDB Data API | `https://data.rcsb.org/rest/v1/core/{entry\|polymer_entity\|assembly}/{ID}` | JSON | None |")
    lines.append("| AlphaFold API | `https://alphafold.ebi.ac.uk/api/prediction/{UniProt_Accession}` | JSON | None |")
    lines.append("| AlphaFold CIF | URL from API `cifUrl` field (currently v6) | mmCIF | None |")
    lines.append("")
    lines.append("**CIF sequence extraction**: Biopython `MMCIF2Dict` was used to parse mmCIF files.")
    lines.append("Target protein residues in chimeric PDB structures were identified via `_struct_ref.pdbx_db_accession`")
    lines.append("and coverage coordinates from `_struct_ref_seq.db_align_beg/end`.")
    lines.append("")

    # 8. Caveats
    lines.append("## 8. Caveats and Limitations")
    lines.append("")
    chimera_pdbs = [r["PDB_ID"] for r in pdb_seq_records if r["is_chimera"]]
    if chimera_pdbs:
        lines.append(f"- **Chimeric PDB structures** ({', '.join(chimera_pdbs)}): These X-ray crystal structures use fusion proteins")
        lines.append("  inserted into the target protein to aid crystallization (standard GPCR crystallography practice).")
        lines.append("  The Excel file records target-protein-only residues after removing the fusion partner.")
    lines.append("- **C-terminal disordered regions**: PDB structures often lack C-terminal residues that are disordered in solution.")
    lines.append("  Only AlphaFold provides full-length predictions.")
    lines.append("- **AlphaFold model version**: The current version is resolved dynamically via the AlphaFold API (`latestVersion` field).")
    lines.append("  Hardcoded version numbers (e.g., v4) may become outdated.")
    lines.append("- **Unreviewed UniProt entries**: TrEMBL entries may be redundant submissions with identical sequences to reviewed Swiss-Prot entries.")
    lines.append("- **PDB JSON DOI fields**: `rcsb_primary_citation.pdbx_database_id_doi` may be empty; use PubMed ID instead.")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append(f"*Report generated automatically by the `protein-human-mouse-struct-collector` skill on {today}.*")

    md_path = os.path.join(output_dir, f"{gene}_Human_Mouse_Data_Report.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Report saved: {md_path}  ({os.path.getsize(md_path)/1024:.1f} KB)")
    return md_path

generate_markdown_report(GENE, df_uniprot, pdb_seq_records, af_seq_records, OUTPUT_DIR)


## 15. Final output summary

In [ ]:
print(f"=== {GENE} Human & Mouse Structure Data Collection — COMPLETE ===")
print(f"\nOutput directory: {os.path.abspath(OUTPUT_DIR)}")
print(f"\n--- CIF files ({len(os.listdir(CIF_DIR))}) ---")
for f in sorted(os.listdir(CIF_DIR)):
    print(f"  {f}  ({os.path.getsize(os.path.join(CIF_DIR, f))/1024:.1f} KB)")

print(f"\n--- JSON files ({len(os.listdir(JSON_DIR))}) ---")
for f in sorted(os.listdir(JSON_DIR)):
    print(f"  {f}  ({os.path.getsize(os.path.join(JSON_DIR, f))/1024:.1f} KB)")

print(f"\n--- Excel & other files ---")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        print(f"  {f}  ({os.path.getsize(fpath)/1024:.1f} KB)")
